In [1]:
import json, os, re, sys
from dataclasses import dataclass, asdict
from typing import Optional
import urllib.request
import urllib.error

In [4]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('Loaded GEMINI_API_KEY from Colab Secrets')
except Exception:
    GEMINI_API_KEY = "YOUR_API_KEY_HERE"
    print('Could not load from Secrets. Set GEMINI_API_KEY manually.')

GEMINI_MODEL = "gemini-3-flash-preview"

GEMINI_URL = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
)

print(f"LLM backend: {GEMINI_MODEL}")

Loaded GEMINI_API_KEY from Colab Secrets
LLM backend: gemini-3-flash-preview


In [5]:
#DATA MODEL

In [6]:
@dataclass
class Resource:
    resource_id: str
    cpu_avg: float
    cpu_p95: float
    memory_avg: float
    network_pct: float
    internet_facing: bool
    identity_attached: bool
    disk_usage_pct: float = 0.0
    error_rate_pct: float = 0.0
    uptime_days: int = 0

@dataclass
class AnalysisResult:
    resource_id: str
    is_anomalous: bool
    anomaly_type: str
    reason: str
    suggested_action: str
    confidence: float
    security_note: Optional[str] = None

print('Data models defined')

Data models defined


In [7]:
# Rule based engine

In [8]:
class RuleEngine:
    """
    Fast, deterministic rule-based anomaly detection.
    Acts as a first-pass classifier and confidence anchor.
    """
    THRESHOLDS = {
        'cpu_high': 85, 'cpu_low': 5, 'cpu_p95_spike': 90,
        'memory_high': 85, 'memory_low': 15,
        'network_high': 80, 'disk_high': 90, 'error_rate_high': 5,
    }

    def analyze(self, r: Resource) -> dict:
        flags = []

        # CPU
        if r.cpu_avg < self.THRESHOLDS['cpu_low'] and r.cpu_p95 < 20:
            flags.append(('over_provisioned', 0.80, f'CPU avg={r.cpu_avg}% and p95={r.cpu_p95}% are very low'))
        if r.cpu_avg > self.THRESHOLDS['cpu_high']:
            flags.append(('cpu_saturation', 0.85, f'CPU avg={r.cpu_avg}% exceeds safe threshold'))
        if r.cpu_p95 > self.THRESHOLDS['cpu_p95_spike'] and r.cpu_avg < 50:
            flags.append(('cpu_spikes', 0.70, f'CPU p95={r.cpu_p95}% with avg={r.cpu_avg}% — bursty workload'))

        # Memory
        if r.memory_avg > self.THRESHOLDS['memory_high']:
            flags.append(('memory_pressure', 0.80, f'Memory avg={r.memory_avg}% is critically high'))
        if r.memory_avg < self.THRESHOLDS['memory_low'] and r.cpu_avg < self.THRESHOLDS['cpu_low']:
            flags.append(('over_provisioned', 0.75, 'Both CPU and memory are very low — resource is likely idle'))

        # Network
        if r.network_pct > self.THRESHOLDS['network_high']:
            flags.append(('high_network_traffic', 0.72, f'Network utilization={r.network_pct}% is high'))

        # Disk
        if r.disk_usage_pct > self.THRESHOLDS['disk_high']:
            flags.append(('disk_saturation', 0.85, f'Disk usage={r.disk_usage_pct}% is critically high'))

        # Error rate
        if r.error_rate_pct > self.THRESHOLDS['error_rate_high']:
            flags.append(('high_error_rate', 0.90, f'Error rate={r.error_rate_pct}% is abnormal'))

        # Imbalance
        if r.cpu_avg > 70 and r.memory_avg < 20:
            flags.append(('resource_imbalance', 0.65, f'High CPU ({r.cpu_avg}%) but very low memory ({r.memory_avg}%)'))
        if r.memory_avg > 80 and r.cpu_avg < 10:
            flags.append(('resource_imbalance', 0.68, f'High memory ({r.memory_avg}%) but idle CPU ({r.cpu_avg}%) — possible leak'))

        if not flags:
            return {'is_anomalous': False, 'anomaly_type': 'normal', 'flags': [], 'confidence': 0.95}

        op_flags    = [f for f in flags if f[0] == 'over_provisioned']
        other_flags = [f for f in flags if f[0] != 'over_provisioned']
        merged = []
        if op_flags:
            best_op = max(op_flags, key=lambda x: x[1])
            reasons = ' | '.join(set(f[2] for f in op_flags))
            merged.append(('over_provisioned', best_op[1], reasons))
        merged.extend(other_flags)

        primary    = max(merged, key=lambda x: x[1])
        confidence = min(primary[1] + 0.05 * (len(merged) - 1), 0.97)

        return {
            'is_anomalous': True,
            'anomaly_type': primary[0],
            'flags': merged,
            'confidence': round(confidence, 2),
        }

    def security_check(self, r: Resource) -> Optional[str]:
        notes = []
        if r.internet_facing and r.identity_attached:
            notes.append('Internet-facing with IAM identity attached — elevated blast radius')
        if r.internet_facing and r.network_pct > 70:
            notes.append('Internet-facing with high outbound network — possible data exfiltration')
        if r.internet_facing and r.error_rate_pct > 5:
            notes.append('Internet-facing with high error rate — may indicate probing or attack')
        if not r.internet_facing and r.network_pct > 80:
            notes.append('Internal resource with high network traffic — possible lateral movement')
        return ' | '.join(notes) if notes else None

print('Rule engine defined')

Rule engine defined


In [9]:
#LLM - GOOGLE gemini

In [10]:
class LLMReasoner:
    """
    Calls Google Gemini to enrich rule-based findings with nuanced
    explanations and fine-tuned confidence scores.
    Falls back gracefully if GEMINI_API_KEY is not set or API fails.
    """

    def __init__(self):
        self.available = bool(GEMINI_API_KEY)
        if not self.available:
            print('⚠️  GEMINI_API_KEY not set — LLM enrichment disabled.')
            print('    Add it via Colab Secrets (🔑) or paste it in Cell 1.')

    def _call_gemini(self, prompt: str) -> Optional[str]:
        payload = json.dumps({
            'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {
                'temperature': 0.2,
                'maxOutputTokens': 600,
                'responseMimeType': 'application/json',
            },
        }).encode('utf-8')

        req = urllib.request.Request(
            GEMINI_URL, data=payload,
            headers={'Content-Type': 'application/json'}, method='POST',
        )
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                data = json.loads(resp.read())
                return data['candidates'][0]['content']['parts'][0]['text']
        except urllib.error.HTTPError as e:
            body = e.read().decode('utf-8', errors='replace')
            print(f'  Gemini HTTP {e.code}: {body[:300]}')
            self.available = False
            return None
        except Exception as e:
            print(f'  Gemini error: {e}')
            self.available = False
            return None

    def enrich(self, r: Resource, rule_result: dict) -> Optional[dict]:
        if not self.available:
            return None

        prompt = f"""You are an expert cloud infrastructure SRE analyst.
Given resource metrics and a rule-based anomaly analysis, provide a nuanced explanation.

Resource metrics:
{json.dumps(asdict(r), indent=2)}

Rule-based analysis:
- is_anomalous: {rule_result['is_anomalous']}
- anomaly_type: {rule_result['anomaly_type']}
- rule_confidence: {rule_result['confidence']}
- signals: {[f[2] for f in rule_result.get('flags', [])]}

Return ONLY a valid JSON object with these exact keys:
{{
  "reason": "2-3 sentence explanation of what is happening and why it matters",
  "suggested_action": "specific, prioritised, actionable recommendation",
  "confidence_adjustment": <float between -0.15 and +0.15>,
  "additional_context": "any nuance the rules might have missed, or empty string"
}}"""

        raw = self._call_gemini(prompt)
        if not raw:
            return None
        clean = re.sub(r'```json|```', '', raw).strip()
        try:
            return json.loads(clean)
        except json.JSONDecodeError:
            return None

print(' Gemini LLM reasoner defined')

 Gemini LLM reasoner defined


In [11]:
class AnomalyDetector:
    """Combines deterministic rules with Gemini-quality explanations."""

    def __init__(self, use_llm: bool = True):
        self.rules = RuleEngine()
        self.llm   = LLMReasoner() if use_llm else None

    def _fallback_reason(self, rule_result: dict, r: Resource) -> tuple:
        atype     = rule_result['anomaly_type']
        flag_text = ' | '.join(f[2] for f in rule_result.get('flags', []))
        reasons = {
            'over_provisioned':    (f'Resource is significantly under-utilized. {flag_text}.',
                                    'Downsize to a smaller instance or implement auto-scaling.'),
            'cpu_saturation':      (f'CPU is heavily loaded. {flag_text}.',
                                    'Scale up CPU, add horizontal scaling, or profile for hotspots.'),
            'cpu_spikes':          (f'CPU shows bursty behavior. {flag_text}.',
                                    'Enable burstable instance or add p95-based auto-scaling.'),
            'memory_pressure':     (f'Memory is critically high. {flag_text}. Risk of OOM kills.',
                                    'Upgrade memory, investigate leaks, or add swap as short-term fix.'),
            'high_network_traffic':(f'Network utilization is high. {flag_text}.',
                                    'Check for unexpected transfers, enable compression, upgrade tier.'),
            'disk_saturation':     (f'Disk is nearly full. {flag_text}. Risk of service failure.',
                                    'Clean logs/temp files, expand volume, or archive old data.'),
            'high_error_rate':     (f'Error rate is abnormally high. {flag_text}.',
                                    'Investigate logs, check dependencies, consider rollback.'),
            'resource_imbalance':  (f'Resource allocation is imbalanced. {flag_text}.',
                                    'Review instance family; try memory-optimized or compute-optimized.'),
            'normal':              ('Resource metrics are within acceptable ranges.',
                                    'No action required. Continue monitoring.'),
        }
        return reasons.get(atype, (f'Anomaly: {flag_text}', 'Investigate and remediate.'))

    def analyze(self, resource_data: dict) -> AnalysisResult:
        r = Resource(
            resource_id       = resource_data.get('resource_id', 'unknown'),
            cpu_avg           = resource_data.get('cpu_avg', 0),
            cpu_p95           = resource_data.get('cpu_p95', 0),
            memory_avg        = resource_data.get('memory_avg', 0),
            network_pct       = resource_data.get('network_pct', 0),
            internet_facing   = resource_data.get('internet_facing', False),
            identity_attached = resource_data.get('identity_attached', False),
            disk_usage_pct    = resource_data.get('disk_usage_pct', 0.0),
            error_rate_pct    = resource_data.get('error_rate_pct', 0.0),
            uptime_days       = resource_data.get('uptime_days', 0),
        )
        rule_result   = self.rules.analyze(r)
        security_note = self.rules.security_check(r)

        llm_result = self.llm.enrich(r, rule_result) if self.llm else None

        if llm_result:
            reason = llm_result.get('reason', '')
            action = llm_result.get('suggested_action', '')
            extra  = llm_result.get('additional_context', '')
            adj    = float(llm_result.get('confidence_adjustment', 0))
            if extra:
                reason = f'{reason} {extra}'.strip()
            confidence = round(min(max(rule_result['confidence'] + adj, 0.05), 0.99), 2)
        else:
            reason, action = self._fallback_reason(rule_result, r)
            confidence     = rule_result['confidence']

        return AnalysisResult(
            resource_id      = r.resource_id,
            is_anomalous     = rule_result['is_anomalous'],
            anomaly_type     = rule_result['anomaly_type'],
            reason           = reason,
            suggested_action = action,
            confidence       = confidence,
            security_note    = security_note,
        )

    def analyze_batch(self, resources: list) -> list:
        return [asdict(self.analyze(res)) for res in resources]

print('AnomalyDetector defined')

AnomalyDetector defined


In [12]:
#Sample data

In [13]:
SAMPLE_RESOURCES = [
    {   # i-1: Over-provisioned, internet-facing with identity
        'resource_id': 'i-1', 'cpu_avg': 2,  'cpu_p95': 5,
        'memory_avg': 70,  'network_pct': 10,
        'internet_facing': True,  'identity_attached': True,
        'disk_usage_pct': 30.0,  'error_rate_pct': 0.1,  'uptime_days': 45,
    },
    {   # i-2: CPU saturated
        'resource_id': 'i-2', 'cpu_avg': 85, 'cpu_p95': 98,
        'memory_avg': 40,  'network_pct': 60,
        'internet_facing': False, 'identity_attached': False,
        'disk_usage_pct': 55.0,  'error_rate_pct': 0.5,  'uptime_days': 12,
    },
    {   # i-3: Bursty CPU
        'resource_id': 'i-3', 'cpu_avg': 45, 'cpu_p95': 92,
        'memory_avg': 60,  'network_pct': 30,
        'internet_facing': False, 'identity_attached': True,
        'disk_usage_pct': 40.0,  'error_rate_pct': 0.2,  'uptime_days': 7,
    },
    {   # i-4: Fully idle
        'resource_id': 'i-4', 'cpu_avg': 3,  'cpu_p95': 8,
        'memory_avg': 12,  'network_pct': 5,
        'internet_facing': True,  'identity_attached': False,
        'disk_usage_pct': 20.0,  'error_rate_pct': 0.0,  'uptime_days': 90,
    },
    {   # i-5: Critical multi-signal failure
        'resource_id': 'i-5', 'cpu_avg': 70, 'cpu_p95': 88,
        'memory_avg': 92,  'network_pct': 85,
        'internet_facing': True,  'identity_attached': True,
        'disk_usage_pct': 93.0,  'error_rate_pct': 8.5,  'uptime_days': 3,
    },
    {   # i-6: Memory leak pattern
        'resource_id': 'i-6', 'cpu_avg': 6,  'cpu_p95': 12,
        'memory_avg': 89,  'network_pct': 15,
        'internet_facing': False, 'identity_attached': True,
        'disk_usage_pct': 45.0,  'error_rate_pct': 0.3,  'uptime_days': 60,
    },
    {   # i-7: Healthy baseline
        'resource_id': 'i-7', 'cpu_avg': 50, 'cpu_p95': 55,
        'memory_avg': 55,  'network_pct': 40,
        'internet_facing': False, 'identity_attached': False,
        'disk_usage_pct': 50.0,  'error_rate_pct': 0.8,  'uptime_days': 30,
    },
]
print(f'{len(SAMPLE_RESOURCES)} sample resources loaded')

7 sample resources loaded


In [14]:
USE_LLM = True

print('=' * 62)
print('  SentnelOps Anomaly Detection System')
print(f'  LLM  : {"Google Gemini (" + GEMINI_MODEL + ")" if USE_LLM else "Disabled"}')
print(f'  Mode : {"Hybrid (Rules + Gemini)" if USE_LLM else "Rule-Based Only"}')
print('=' * 62)

detector = AnomalyDetector(use_llm=USE_LLM)
results  = detector.analyze_batch(SAMPLE_RESOURCES)

for r in results:
    status = '🔴 ANOMALOUS' if r['is_anomalous'] else '🟢 NORMAL'
    print(f"\n{status}  [{r['resource_id']}]  type={r['anomaly_type']}  confidence={r['confidence']}")
    print(f"  Reason : {r['reason']}")
    print(f"  Action : {r['suggested_action']}")
    if r.get('security_note'):
        print(f"  ⚠ Sec  : {r['security_note']}")

  SentnelOps Anomaly Detection System
  LLM  : Google Gemini (gemini-3-flash-preview)
  Mode : Hybrid (Rules + Gemini)
  Gemini HTTP 503: {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}


🔴 ANOMALOUS  [i-1]  type=over_provisioned  confidence=0.8
  Reason : Resource is significantly under-utilized. CPU avg=2% and p95=5% are very low.
  Action : Downsize to a smaller instance or implement auto-scaling.
  ⚠ Sec  : Internet-facing with IAM identity attached — elevated blast radius

🟢 NORMAL  [i-2]  type=normal  confidence=0.95
  Reason : Resource metrics are within acceptable ranges.
  Action : No action required. Continue monitoring.

🔴 ANOMALOUS  [i-3]  type=cpu_spikes  confidence=0.7
  Reason : CPU shows bursty behavior. CPU p95=92% with avg=45% — bursty workload.
  Action : Enable burstable instance or add p95-based auto-scaling.

🔴 ANOMA

In [15]:
import json

output_path = 'sample_outputs.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {output_path}')
print()
print(json.dumps(results, indent=2))

Saved to sample_outputs.json

[
  {
    "resource_id": "i-1",
    "is_anomalous": true,
    "anomaly_type": "over_provisioned",
    "reason": "Resource is significantly under-utilized. CPU avg=2% and p95=5% are very low.",
    "suggested_action": "Downsize to a smaller instance or implement auto-scaling.",
    "confidence": 0.8,
    "security_note": "Internet-facing with IAM identity attached \u2014 elevated blast radius"
  },
  {
    "resource_id": "i-2",
    "is_anomalous": false,
    "anomaly_type": "normal",
    "reason": "Resource metrics are within acceptable ranges.",
    "suggested_action": "No action required. Continue monitoring.",
    "confidence": 0.95,
    "security_note": null
  },
  {
    "resource_id": "i-3",
    "is_anomalous": true,
    "anomaly_type": "cpu_spikes",
    "reason": "CPU shows bursty behavior. CPU p95=92% with avg=45% \u2014 bursty workload.",
    "suggested_action": "Enable burstable instance or add p95-based auto-scaling.",
    "confidence": 0.7,
    "

In [16]:
from google.colab import files
files.download('sample_outputs.json')
print('Download started!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!


In [18]:
#testing my own resource

In [17]:
my_resource = {
    'resource_id':      'my-server-1',
    'cpu_avg':          15,        # average CPU %
    'cpu_p95':          75,        # p95 CPU %
    'memory_avg':       60,        # average memory %
    'network_pct':      20,        # network utilization %
    'internet_facing':  True,      # exposed to internet?
    'identity_attached':False,     # IAM role/identity attached?
    'disk_usage_pct':   45.0,      # disk usage %
    'error_rate_pct':   1.2,       # application error rate %
    'uptime_days':      14,        # how long has it been running
}

result = detector.analyze(my_resource)
print(json.dumps(asdict(result), indent=2))

{
  "resource_id": "my-server-1",
  "is_anomalous": false,
  "anomaly_type": "normal",
  "reason": "Resource metrics are within acceptable ranges.",
  "suggested_action": "No action required. Continue monitoring.",
  "confidence": 0.95,
  "security_note": null
}
